# Soft Actor Critic in Parallel on Cloud pendulum

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib
import random
import time

from IPython.display import HTML

from pendulum_plant.pendulum_plant import PendulumPlant
from pendulum_plant.simulation import Simulator

from sac.sac_parallel_disconnected_hw import sac_trainer
from sac.sac_controller import SacController

# Import user token from .env file (Alternatively set USER_TOKEN = "your token")
from dotenv import load_dotenv
load_dotenv()
USER_TOKEN = os.getenv("USER_TOKEN")

In [4]:
# pendulum parameters
mass = 0.06
length = 0.1
damping = 0.0004
torque_limit = 0.02
coulomb_fric = 0.0
inertia = mass*length**2
gravity = 9.81


# environment parameters
dt = 0.02
max_steps = 500 
reward_type = "combined_reward"
target = [np.pi, 0]
target_epsilon = [0.1, 0.1]
random_init = "everywhere"
random_init_eval = "False"
dt_step_scaling=0.80

# Training

In [ ]:
# training parameters
log_dir = "log_data/sac_hw/run_1"
learning_rate = 0.0003
n_episodes=200
gradient_steps=1000
batch_size=1024
eval_every = 5
eval_episodes = 1
start_training = 2000
number_of_envs = 1
reward_limit = 5_000

trainer = sac_trainer(log_dir=log_dir, verbose=0)

trainer.init_environment(user_token=USER_TOKEN,
                         dt=dt,
                         max_steps=max_steps,
                         reward_type=reward_type,
                         state_representation=3,
                         target=target, 
                         state_target_epsilon=target_epsilon,
                         random_init=random_init,
                         random_init_eval=random_init_eval,
                         dt_step_scaling=dt_step_scaling)

trainer.init_agent(learning_rate=learning_rate,
                   warm_start=False,
                   )

trainer.train(n_episodes=n_episodes,
              gradient_steps=gradient_steps,
              batch_size=batch_size,
              eval_every=eval_every,
              eval_episodes=eval_episodes,
              start_training=start_training,
              save_path=log_dir,
              number_of_envs=number_of_envs,
              reward_limit=reward_limit)



In [ ]:
# Evaluation training loop
base_log_dir = "log_data/evaluation_run_1envs"
learning_rate = 0.0003
n_episodes=200
gradient_steps=1000
batch_size=1024
eval_every = 5
eval_episodes = 1
start_training = 2000
reward_limit = 5_000

def run_evaluation(iters=5):
    for run_idx in range(0,iters):
        print(f"Running evaluation run {run_idx}")
        log_dir = os.path.join(base_log_dir, f"run_{run_idx}")
        trainer = sac_trainer(log_dir=log_dir, verbose=0)
        
        trainer.init_environment(user_token=USER_TOKEN,
                                 dt=dt,
                                 max_steps=max_steps,
                                 reward_type=reward_type,
                                 state_representation=3,
                                 target=target, 
                                 state_target_epsilon=target_epsilon,
                                 random_init=random_init,
                                 random_init_eval=random_init_eval,
                                 dt_step_scaling=dt_step_scaling)
        
        trainer.init_agent(learning_rate=learning_rate,
                           warm_start=False,
                           #warm_start_path="best_model/best_model_tl04_dt02_80ep_working_trainedHW.zip"
                           )
        
        trainer.train(n_episodes=n_episodes,
                      gradient_steps=gradient_steps,
                      batch_size=batch_size,
                      eval_every=eval_every,
                      eval_episodes=eval_episodes,
                      start_training=start_training,
                      save_path=log_dir,
                      number_of_envs=number_of_envs,
                      reward_limit=reward_limit)

base_log_dir = "log_data/hw_no_env_eval_141025/sac_parallel_evaluation_env_4"
number_of_envs = 4
run_evaluation()

#base_log_dir = "log_data/hw_no_env_eval_141025/sac_parallel_evaluation_env_3"
#number_of_envs = 3
#run_evaluation()

#base_log_dir = "log_data/hw_no_env_eval_141025/sac_parallel_evaluation_env_2"
#number_of_envs = 2
#run_evaluation()

#base_log_dir = "log_data/hw_no_env_eval_141025/sac_parallel_evaluation_env_1"
#number_of_envs = 1
#run_evaluation()

In [ ]:
# Create a video showing training progress

from sac.sac_analysis_utils import create_evaluation_progress_video

folder_path = "log_data/sac_hw/run_1/evaluation_videos"
create_evaluation_progress_video(folder_path)

# Simulate swing using SAC controller and Gym environment

In [ ]:
# initialize the pendulum
pendulum = PendulumPlant(mass=mass,
                         length=length,
                         damping=damping,
                         gravity=gravity,
                         coulomb_fric=coulomb_fric,
                         inertia=inertia,
                         torque_limit=torque_limit)

sim = Simulator(plant=pendulum)

# get the controller we trained earlier
#model_path = "log_data/no_env_eval_180925_gs_2000_stop/sac_parallel_evaluation_env_4/run_1/best_model.zip"
model_path = 'log_data/sac_hw/run_1/best_model.zip'

controller = SacController(model_path=model_path,
                           torque_limit=torque_limit,
                           use_symmetry=False,
                           state_representation=3,
                           deterministic=True)

# Run simulated experiment
x0_sim = [0.0, 0.0]
dt = 0.01
t_final = 10
integrator = "runge_kutta"

T, X, U = sim.simulate(t0=0.0,
                                   x0=x0_sim,
                                   tf=t_final,
                                   dt=dt,
                                   controller=controller,
                                   integrator=integrator)


fig, ax = plt.subplots(3, 1, figsize=(18, 6), sharex="all")
print(f"Final torque: {U[-1]}")
ax[0].plot(T, np.asarray(X).T[0], label="theta")
ax[0].hlines(y=[-np.pi, np.pi], xmin=0, xmax = t_final, linestyles = ':', colors='g')
ax[0].set_ylabel("angle [rad]")
ax[0].legend(loc="best")
ax[1].plot(T, np.asarray(X).T[1], label="theta dot")
ax[1].set_ylabel("angular velocity [rad/s]")
ax[1].legend(loc="best")
ax[2].plot(T, np.asarray(U).flatten(), label="u")
ax[2].set_xlabel("time [s]")
ax[2].set_ylabel("input torque [Nm]")
ax[2].hlines(y=0, xmin=0, xmax=t_final, linestyles = ':', colors='g')
ax[2].legend(loc="best")
plt.show()


In [ ]:
# Simulate on hardware using experiment
T,X,U,U_des, vod = pendulum.run_on_hardware(10, 0.01, controller, USER_TOKEN, preparation_time=0.0)


In [ ]:
def plot_timeseries(T, X, U):
    X = np.asarray(X)
    U = np.asarray(U)
    
    fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
    
    # Theta
    axs[0].plot(T, X[:, 0], label="Position")
    #axs[0].plot(T, X_gym[:, 0], label="theta gym", linestyle='-', alpha=0.8)
    axs[0].set_title("Theta")
    axs[0].set_xlabel("Time [s]")
    axs[0].set_ylabel("Angle [rad]")
    axs[0].hlines([-np.pi, np.pi], T[0],T[-1], linestyle="--", alpha=0.3, color="grey")
    axs[0].legend()
    axs[0].grid(True)

    # Theta dot
    axs[1].plot(T, X[:, 1], label="Velocity")
    #axs[1].plot(T, X_gym[:, 1], label="theta dot gym", linestyle="-", alpha=0.8)
    axs[1].set_title("Theta dot")
    axs[1].set_xlabel("Time [s]")
    axs[1].set_ylabel("Angular velocity [rad/s]")
    axs[1].legend()
    axs[1].grid(True)

    # Torque
    axs[2].plot(T, U, label="Torque")
    #axs[2].plot(T[:-1], U_gym * 0.02, label="u gym", linestyle="-", alpha=0.8)
    axs[2].set_title("Torque")
    axs[2].set_xlabel("Time [s]")
    axs[2].set_ylabel("Torque [Nm]")
    axs[2].legend()
    axs[2].grid(True)

    plt.tight_layout()
    plt.show()

plot_timeseries(T,X,U)

In [ ]:
# Sim on hardware using the gym environment

from stable_baselines3 import SAC
from pendulum_plant.gym_environment_hw import SimplePendulumEnv
import numpy as np
import time
from cloudpendulumclient.client import Client


model_path = 'log_data/sac_hw/run_1/best_model.zip'
model = SAC.load(model_path)

def run_test_episode(model):
    c = Client()

    env = SimplePendulumEnv(
        user_token = USER_TOKEN,
        gym_session=False,
        dt=0.02,
        max_steps=500,
        reward_type="combined_reward",
        state_representation=3,
        torque_limit=0.02,
        random_init="False",
        dt_step_scaling = .8,
        #verbal=1
    )
    env.start_gym_session(30, 1)
    obs = env.reset(record=True)

    print("Reset obs: ", obs)
    total_reward = 0
    
    obs_list = [obs]
    torque_list = []
    
    for step in range(1,env.max_steps):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = env.step(action)

        obs_list.append(obs)
        torque_list.append(action)
        
        total_reward += reward
        #print(f"Step {step}, Reward: {reward:.3f}, Done: {done}")

        if done:
            break            
    env.close(save_video=True, video_path="env_vid")
    print("Episode finished. Total reward:", total_reward)
    
    obs_list = np.array(obs_list)
    angles = np.arctan2(obs_list[:, 1], obs_list[:, 0])
    obs_list = np.column_stack((angles, obs_list[:, 2]))
    env.stop_gym_session()
    return obs_list, np.array(torque_list)


X_gym, U_gym = run_test_episode(model)

In [ ]:
def plot_timeseries(T, X, U, X_gym, U_gym):
    X = np.asarray(X)
    U = np.asarray(U)
    
    fig, axs = plt.subplots(1, 3, figsize=(15, 4), sharex=True)
    
    # Theta
    axs[0].plot(T, X[:, 0], label="theta cont")
    axs[0].plot(T, X_gym[:, 0], label="theta gym", linestyle='-', alpha=0.8)
    axs[0].set_title("Theta")
    axs[0].set_xlabel("Time [s]")
    axs[0].set_ylabel("Angle [rad]")
    axs[0].hlines([-np.pi, np.pi], T[0],T[-1], linestyle="--", alpha=0.3, color="grey")
    axs[0].legend()
    axs[0].grid(True)

    # Theta dot
    axs[1].plot(T, X[:, 1], label="theta dot cont")
    axs[1].plot(T, X_gym[:, 1], label="theta dot gym", linestyle="-", alpha=0.8)
    axs[1].set_title("Theta dot")
    axs[1].set_xlabel("Time [s]")
    axs[1].set_ylabel("Angular velocity [rad/s]")
    axs[1].legend()
    axs[1].grid(True)

    # Torque
    axs[2].plot(T, U, label="u cont")
    axs[2].plot(T[:-1], U_gym * 0.02, label="u gym", linestyle="-", alpha=0.8)
    axs[2].set_title("Torque")
    axs[2].set_xlabel("Time [s]")
    axs[2].set_ylabel("Torque [Nm]")
    axs[2].legend()
    axs[2].grid(True)

    plt.tight_layout()
    plt.show()

plot_timeseries(T,X,U, X_gym, U_gym)

# SAC test of both deterministic and non-deterministic policy

In [ ]:


# --- Setup simulation parameters ---
x0_sim = [0.0, 0.0]
dt = 0.02
t_final = 20
integrator = "runge_kutta"
n_stochastic_trials = 4  # Number of stochastic rollouts

# --- Run deterministic rollout ---
controller.deterministic = True
T_det, X_det, U_det = sim.simulate(t0=0.0, x0=x0_sim, tf=t_final, dt=dt,
                                   controller=controller, integrator=integrator)

# --- Run stochastic rollouts ---
stochastic_rollouts = []
controller.deterministic = False
for _ in range(n_stochastic_trials):
    T, X, U = sim.simulate(t0=0.0, x0=x0_sim, tf=t_final, dt=dt,
                           controller=controller, integrator=integrator)
    stochastic_rollouts.append((T, X, U))

# --- Get alpha value if available ---
#try:
#    alpha = float(controller.model.ent_coef)
#except:
#    alpha = "unknown"

# --- Plotting ---
fig, ax = plt.subplots(3, 1, figsize=(18, 10), sharex="all")
#ax[0].set_title(f"Deterministic (top) and {n_stochastic_trials} Stochastic Rollouts (α = {alpha:.3f})")

# Plot deterministic rollout
X_det = np.array(X_det)
U_det = np.array(U_det)
ax[0].plot(T_det, X_det[:, 0], color='black', label="theta (deterministic)", linewidth=2)
ax[0].hlines(y=[-np.pi, np.pi], xmin=0, xmax=t_final, linestyles=':', colors='g')
ax[0].set_ylabel("angle [rad]")
ax[0].legend(loc="upper right")

# Plot stochastic rollouts
colors = plt.cm.viridis(np.linspace(0.3, 1.0, n_stochastic_trials))
for i, (T, X, U) in enumerate(stochastic_rollouts):
    X = np.array(X)
    U = np.array(U)
    ax[1].plot(T, X[:, 0], label=f"θ trial {i+1}", color=colors[i])
    ax[2].plot(T, U.flatten(), label=f"u trial {i+1}", color=colors[i])

ax[1].hlines(y=[-np.pi, np.pi], xmin=0, xmax=t_final, linestyles=':', colors='g')
ax[1].set_ylabel("angle [rad] (stochastic)")
ax[1].legend(loc="best")

ax[2].hlines(y=0, xmin=0, xmax=t_final, linestyles=':', colors='g')
ax[2].set_ylabel("torque [Nm]")
ax[2].set_xlabel("time [s]")
ax[2].legend(loc="best")

plt.tight_layout()
plt.show()


# Policy Visualization

In [ ]:
from sac.sac_analysis_utils import visualize_policy_with_stabilization_view

visualize_policy_with_stabilization_view(controller)

# Replay buffer visualization and analysis

In [ ]:
##### Print out replay buffer
from sac.sac_analysis_utils import visualize_replay_buffer
visualize_replay_buffer('log_data/sac_hw/run_1/replay_buffer.pkl', [0,1000], sim)